In [1]:
%pip install -U \
  "autogluon.timeseries==1.5.0" \
  "torch>=2.6.0" \
  "torchvision>=0.21.0" \
  "torchaudio>=2.6.0" \
  "transformers>=4.48.0" \
  "tokenizers>=0.21.0" \
  "numpy==1.26.4"

  Using cached tokenizers-0.23.1-cp310-abi3-macosx_11_0_arm64.whl.metadata (9.8 kB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
  Using cached torchvision-0.26.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached torchvision-0.25.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.4 kB)

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from sklearn.metrics import r2_score,mean_squared_error

In [3]:
df = pd.read_csv("../data/training/bitola_final_data.csv")
df.head()

,timestamp,sensorId,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,...,neighbor3_wind_speed,season,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2023-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,58.333333,28.333333,13.666667,943.0,12.00,8.3,63.645833,63.645833,...,6.98,winter,0.000000,1.000000,-0.5,0.866025,-0.433884,-0.900969,0,1
1,2023-12-01 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,59.250000,13.750000,5.000000,943.0,12.00,9.3,64.750000,64.750000,...,7.50,winter,0.258819,0.965926,-0.5,0.866025,-0.433884,-0.900969,0,1
2,2023-12-01 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,60.000000,10.000000,4.500000,942.5,12.25,8.3,65.312500,65.312500,...,6.74,winter,0.500000,0.866025,-0.5,0.866025,-0.433884,-0.900969,0,1
3,2023-12-01 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,59.250000,9.750000,4.500000,942.0,13.00,9.4,64.854167,64.854167,...,7.84,winter,0.707107,0.707107,-0.5,0.866025,-0.433884,-0.900969,0,1
4,2023-12-01 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,59.500000,5.000000,2.000000,942.0,13.00,9.0,65.062500,65.062500,...,8.32,winter,0.866025,0.500000,-0.5,0.866025,-0.433884,-0.900969,0,1


In [4]:
df.describe()

,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor1_pressure,...,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
count,192533.000000,142475.00000,142887.000000,192533.000000,192533.000000,192907.000000,192533.000000,192533.000000,192533.000000,192533.000000,...,192907.000000,192907.000000,1.929840e+05,1.929840e+05,192984.000000,1.929840e+05,192984.000000,192984.000000,192984.000000,192984.000000
mean,52.706427,26.40624,14.096921,939.802395,16.796424,6.075865,52.234773,53.623165,51.226443,943.013828,...,6.174232,6.158097,-1.845999e-17,-5.550655e-17,-0.002785,-3.554140e-03,-0.002997,-0.000684,0.287278,0.414501
std,15.739595,49.61000,26.306709,12.804769,9.168444,3.743897,15.784644,16.410186,15.056326,6.656861,...,3.866399,3.849372,7.071086e-01,7.071086e-01,0.706861,7.073415e-01,0.707344,0.706866,0.452493,0.492637
min,9.500000,0.00000,0.000000,869.000000,-12.000000,0.000000,9.500000,9.500000,9.500000,911.000000,...,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-1.000000,-1.000000e+00,-0.974928,-0.900969,0.000000,0.000000
25%,40.750000,5.25000,2.250000,937.000000,9.666667,3.500000,40.354167,41.314815,39.750000,939.000000,...,3.500000,3.500000,-7.071068e-01,-7.071068e-01,-0.500000,-8.660254e-01,-0.781831,-0.900969,0.000000,0.000000
50%,53.562500,10.75000,5.500000,942.000000,16.000000,5.300000,53.000000,54.468750,52.250000,943.000000,...,5.400000,5.400000,6.123234e-17,-6.123234e-17,0.000000,-1.836970e-16,0.000000,-0.222521,0.000000,0.000000
75%,64.937500,26.00000,14.000000,946.000000,23.750000,7.744444,64.250000,66.000000,63.000000,947.000000,...,7.900000,7.900000,7.071068e-01,7.071068e-01,0.866025,5.000000e-01,0.781831,0.623490,1.000000,1.000000
max,99.000000,1995.00000,639.250000,971.750000,47.500000,32.800000,99.000000,99.000000,86.750000,971.750000,...,32.800000,32.800000,1.000000e+00,1.000000e+00,1.000000,1.000000e+00,0.974928,1.000000,1.000000,1.000000


In [5]:
len(df)

192984

In [6]:
df['sensorId'].value_counts()

sensorId
16836a55-7140-43e2-9a63-56fac5cba714    17544
2002                                    17544
23b735ef-a996-4a7f-9998-2aa7e78827b0    17544
2819ecbb-5de3-4092-aa1d-4ba3a8c40add    17544
40f081a6-4095-43f7-bffb-64e2af8c026e    17544
7b316592-8036-41e2-b8dc-b06b6a9afd54    17544
874ff9c6-786d-45fc-a90e-48c7ffe03417    17544
87f82783-853b-417d-8964-b5cf11e44873    17544
d241a044-0a06-40c2-9d90-c91fd0a95060    17544
d7060523-b163-4cc3-bc94-970dac72a38a    17544
fec52a19-9148-4350-a1b4-ae0da05ee199    17544
Name: count, dtype: int64

In [7]:
TARGET = 'pm10'
ID_COL = 'sensorId'
TIME_COL = 'timestamp'
PREDICTION_LENGTH = 512

In [8]:
df[TIME_COL] = pd.to_datetime(df[TIME_COL])

In [9]:
df[TIME_COL] = df[TIME_COL].dt.tz_convert(None)

In [10]:
print(df["timestamp"].dtype)

datetime64[ns]


In [11]:
data = TimeSeriesDataFrame.from_data_frame(
    df,
    id_column=ID_COL,
    timestamp_column=TIME_COL
)

In [12]:
data.columns

Index(['humidity', 'pm10', 'pm25', 'pressure', 'temperature', 'wind_speed',
       'neighbor1_humidity', 'neighbor2_humidity', 'neighbor3_humidity',
       'neighbor1_pressure', 'neighbor2_pressure', 'neighbor3_pressure',
       'neighbor1_temperature', 'neighbor2_temperature',
       'neighbor3_temperature', 'neighbor1_wind_speed', 'neighbor2_wind_speed',
       'neighbor3_wind_speed', 'season', 'hour_sin', 'hour_cos', 'month_sin',
       'month_cos', 'day_sin', 'day_cos', 'is_weekend', 'is_heating_season'],
      dtype='object')

In [13]:
train_data, test_data = data.train_test_split(prediction_length=PREDICTION_LENGTH)

In [14]:
forecast_columns  = [ elem for elem in data.columns if elem not in ['pm10','pm25']]
forecast_columns

['humidity',
 'pressure',
 'temperature',
 'wind_speed',
 'neighbor1_humidity',
 'neighbor2_humidity',
 'neighbor3_humidity',
 'neighbor1_pressure',
 'neighbor2_pressure',
 'neighbor3_pressure',
 'neighbor1_temperature',
 'neighbor2_temperature',
 'neighbor3_temperature',
 'neighbor1_wind_speed',
 'neighbor2_wind_speed',
 'neighbor3_wind_speed',
 'season',
 'hour_sin',
 'hour_cos',
 'month_sin',
 'month_cos',
 'day_sin',
 'day_cos',
 'is_weekend',
 'is_heating_season']

In [15]:
predictor = TimeSeriesPredictor(
    target= TARGET,
    prediction_length=PREDICTION_LENGTH,
    eval_metric="MASE",
    known_covariates_names=forecast_columns
)

In [16]:
predictor.fit(
    train_data,
    presets="chronos2",   # or "chronos2_small" for faster training
    time_limit=300,
    verbosity=2
)

Beginning AutoGluon training... Time limit = 300s
AutoGluon will save models to '/mnt/c/Users/RazorVision/Desktop/project-vrnmp/offline-Phase/AutogluonModels/ag-20260715_114407'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          16
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 12.00/12.00 GB
Total GPU Memory:   Free: 12.00 GB, Allocated: 0.00 GB, Total: 12.00 GB
GPU Count:          1
Memory Avail:       9.57 GB / 15.58 GB (61.4%)
Disk Space Avail:   49.21 GB / 464.89 GB (10.6%)
Setting presets to: chronos2

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MASE,
 'hyperparameters': {'Chronos2': {'model_path': 'autogluon/chronos-2'}},
 'known_covariates_names': ['humidity',
                            'pressure',
                        

In [17]:
forecast_df = pd.read_csv('../data/raw/bitola_forecast_weather.csv')
forecast_df

,timestamp,sensorId,humidity,pressure,temperature,wind_speed
0,2025-11-09 16:00:00,16836a55-7140-43e2-9a63-56fac5cba714,88,944.7,12.5,7.1
1,2025-11-09 17:00:00,16836a55-7140-43e2-9a63-56fac5cba714,90,944.7,12.1,7.1
2,2025-11-09 18:00:00,16836a55-7140-43e2-9a63-56fac5cba714,91,944.5,12.0,6.2
3,2025-11-09 19:00:00,16836a55-7140-43e2-9a63-56fac5cba714,91,944.9,12.0,5.8
4,2025-11-09 20:00:00,16836a55-7140-43e2-9a63-56fac5cba714,93,944.8,11.8,5.8
...,...,...,...,...,...,...
59285,2026-03-01 18:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,51,950.9,7.6,7.4
59286,2026-03-01 19:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,56,950.7,6.5,6.2
59287,2026-03-01 20:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,56,951.1,5.8,6.8
59288,2026-03-01 21:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,58,951.2,5.2,6.9


In [18]:
neighbourhood_matrix = pd.read_csv('../data/neighbors_data/bitola_sensor_distances.csv')

In [19]:
valid_sensors = df['sensorId'].unique()

In [20]:
neighbors_df_clean = neighbourhood_matrix[
    (neighbourhood_matrix['sensor_id'].isin(valid_sensors)) &
    (neighbourhood_matrix['neighbor_id'].isin(valid_sensors))
].copy()

In [21]:
def append_neighbors(df_hourly, neighbors_df, weather_cols, k_search=20, k_keep=3):
    # 1. Standardize the neighbor list
    # Ensure we only take the top K based on distance
    neighbors_topk = (
        neighbors_df.sort_values(["sensor_id", "distance_km"])
        .groupby("sensor_id")
        .head(k_search)
        .copy()
    )

    # Track original distance rank
    neighbors_topk['dist_rank'] = neighbors_topk.groupby("sensor_id").cumcount() + 1

    # 2. Merge with main data
    # We use 'neighbor_id' from the matrix to match 'sensorId' in the hourly data
    neighbor_values = neighbors_topk.merge(
        df_hourly[['sensorId', 'timestamp'] + weather_cols],
        left_on='neighbor_id',
        right_on='sensorId',
        how='inner'
    )

    # 3. Filter for availability
    # The 'sensor_id' here is the ORIGINAL sensor we are finding neighbors for
    available_topk = (
        neighbor_values.sort_values(['sensor_id', 'timestamp', 'dist_rank'])
        .groupby(['sensor_id', 'timestamp'])
        .head(k_keep)
        .copy()
    )

    # Create the 1, 2, 3 rank for the wide-format columns
    available_topk['final_rank'] = available_topk.groupby(['sensor_id', 'timestamp']).cumcount() + 1

    # 4. Pivot to wide format
    pivot_df = available_topk.pivot(
        index=['sensor_id', 'timestamp'],
        columns='final_rank',
        values=weather_cols
    )

    # Clean up column names: neighbor1_temp, neighbor2_temp, etc.
    if isinstance(pivot_df.columns, pd.MultiIndex):
        pivot_df.columns = [f"neighbor{rank}_{col}" for col, rank in pivot_df.columns]
    else:
        # Handle case with only one weather column
        pivot_df.columns = [f"neighbor{i}_{weather_cols[0]}" for i in pivot_df.columns]

    pivot_df = pivot_df.reset_index()

    # 5. Final Join back to original data
    df_result = df_hourly.merge(
        pivot_df,
        left_on=['sensorId', 'timestamp'],
        right_on=['sensor_id', 'timestamp'],
        how='left'
    ).drop(columns=['sensor_id'])

    return df_result

In [22]:
weather_cols = ['humidity', 'pressure', 'temperature', 'wind_speed']

In [23]:
forecast_df = append_neighbors(forecast_df,neighbors_df_clean, weather_cols)
forecast_df

,timestamp,sensorId,humidity,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor1_pressure,neighbor2_pressure,neighbor3_pressure,neighbor1_temperature,neighbor2_temperature,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed
0,2025-11-09 16:00:00,16836a55-7140-43e2-9a63-56fac5cba714,88,944.7,12.5,7.1,92.0,88.0,92.0,942.4,944.9,945.4,11.9,12.5,12.1,4.0,7.1,4.0
1,2025-11-09 17:00:00,16836a55-7140-43e2-9a63-56fac5cba714,90,944.7,12.1,7.1,87.0,90.0,87.0,942.4,945.0,945.4,11.8,12.1,12.0,2.8,7.1,2.8
2,2025-11-09 18:00:00,16836a55-7140-43e2-9a63-56fac5cba714,91,944.5,12.0,6.2,93.0,91.0,93.0,942.2,944.8,945.2,11.4,12.0,11.6,3.9,6.2,3.9
3,2025-11-09 19:00:00,16836a55-7140-43e2-9a63-56fac5cba714,91,944.9,12.0,5.8,95.0,91.0,95.0,942.6,945.1,945.5,11.3,12.0,11.5,2.9,5.8,2.9
4,2025-11-09 20:00:00,16836a55-7140-43e2-9a63-56fac5cba714,93,944.8,11.8,5.8,98.0,93.0,98.0,942.5,945.0,945.5,11.1,11.8,11.3,3.8,5.8,3.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59285,2026-03-01 18:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,51,950.9,7.6,7.4,44.0,51.0,44.0,953.1,949.3,953.3,7.9,7.5,7.9,5.9,7.4,5.9
59286,2026-03-01 19:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,56,950.7,6.5,6.2,52.0,56.0,52.0,953.0,949.1,953.2,7.1,6.4,7.1,4.5,6.2,4.5
59287,2026-03-01 20:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,56,951.1,5.8,6.8,56.0,56.0,56.0,953.4,949.4,953.6,6.2,5.7,6.2,4.9,6.8,4.9
59288,2026-03-01 21:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,58,951.2,5.2,6.9,59.0,58.0,59.0,953.6,949.6,953.8,5.4,5.1,5.4,4.6,6.9,4.6


In [24]:
def extract_time_features(df, timestamp_col='timestamp'):
    
    month = df[timestamp_col].dt.month

    df['season'] = np.select(
        [
            month.isin([12, 1, 2]),
            month.isin([3, 4, 5]),
            month.isin([6, 7, 8]),
            month.isin([9, 10, 11])
        ],
        [
            'winter',
            'spring',
            'summer',
            'autumn'
        ]
    )

    df['hour_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.hour / 24)


    df['month_sin'] = np.sin(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)
    
    df['day_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.dayofweek / 7)
    df['day_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.dayofweek / 7)


    df['is_weekend'] = df[timestamp_col].dt.dayofweek.isin([5, 6]).astype(int)

    df['is_heating_season'] = df[timestamp_col].dt.month.isin([11, 12, 1, 2, 3]).astype(int)

    return df

In [25]:
forecast_df['timestamp'] = pd.to_datetime(forecast_df['timestamp'])
forecast_df = extract_time_features(forecast_df)
forecast_df

,timestamp,sensorId,humidity,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor1_pressure,...,neighbor3_wind_speed,season,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2025-11-09 16:00:00,16836a55-7140-43e2-9a63-56fac5cba714,88,944.7,12.5,7.1,92.0,88.0,92.0,942.4,...,4.0,autumn,-0.866025,-5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
1,2025-11-09 17:00:00,16836a55-7140-43e2-9a63-56fac5cba714,90,944.7,12.1,7.1,87.0,90.0,87.0,942.4,...,2.8,autumn,-0.965926,-2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
2,2025-11-09 18:00:00,16836a55-7140-43e2-9a63-56fac5cba714,91,944.5,12.0,6.2,93.0,91.0,93.0,942.2,...,3.9,autumn,-1.000000,-1.836970e-16,-0.866025,0.5,-0.781831,0.62349,1,1
3,2025-11-09 19:00:00,16836a55-7140-43e2-9a63-56fac5cba714,91,944.9,12.0,5.8,95.0,91.0,95.0,942.6,...,2.9,autumn,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
4,2025-11-09 20:00:00,16836a55-7140-43e2-9a63-56fac5cba714,93,944.8,11.8,5.8,98.0,93.0,98.0,942.5,...,3.8,autumn,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59285,2026-03-01 18:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,51,950.9,7.6,7.4,44.0,51.0,44.0,953.1,...,5.9,spring,-1.000000,-1.836970e-16,0.866025,0.5,-0.781831,0.62349,1,1
59286,2026-03-01 19:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,56,950.7,6.5,6.2,52.0,56.0,52.0,953.0,...,4.5,spring,-0.965926,2.588190e-01,0.866025,0.5,-0.781831,0.62349,1,1
59287,2026-03-01 20:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,56,951.1,5.8,6.8,56.0,56.0,56.0,953.4,...,4.9,spring,-0.866025,5.000000e-01,0.866025,0.5,-0.781831,0.62349,1,1
59288,2026-03-01 21:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,58,951.2,5.2,6.9,59.0,58.0,59.0,953.6,...,4.6,spring,-0.707107,7.071068e-01,0.866025,0.5,-0.781831,0.62349,1,1


In [26]:
start = pd.Timestamp("2025-11-09 16:00:00")
end = pd.Timestamp("2025-12-01 00:00:00")

filtered_df = forecast_df[
    (forecast_df["timestamp"] >= start) &
    (forecast_df["timestamp"] < end)
]

In [27]:
filtered_df

,timestamp,sensorId,humidity,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor1_pressure,...,neighbor3_wind_speed,season,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2025-11-09 16:00:00,16836a55-7140-43e2-9a63-56fac5cba714,88,944.7,12.5,7.1,92.0,88.0,92.0,942.4,...,4.0,autumn,-0.866025,-5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
1,2025-11-09 17:00:00,16836a55-7140-43e2-9a63-56fac5cba714,90,944.7,12.1,7.1,87.0,90.0,87.0,942.4,...,2.8,autumn,-0.965926,-2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
2,2025-11-09 18:00:00,16836a55-7140-43e2-9a63-56fac5cba714,91,944.5,12.0,6.2,93.0,91.0,93.0,942.2,...,3.9,autumn,-1.000000,-1.836970e-16,-0.866025,0.5,-0.781831,0.62349,1,1
3,2025-11-09 19:00:00,16836a55-7140-43e2-9a63-56fac5cba714,91,944.9,12.0,5.8,95.0,91.0,95.0,942.6,...,2.9,autumn,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
4,2025-11-09 20:00:00,16836a55-7140-43e2-9a63-56fac5cba714,93,944.8,11.8,5.8,98.0,93.0,98.0,942.5,...,3.8,autumn,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57102,2025-11-30 19:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,88,940.4,4.3,5.3,92.0,88.0,92.0,942.9,...,0.8,autumn,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
57103,2025-11-30 20:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,85,940.9,4.1,5.0,91.0,85.0,91.0,943.4,...,2.1,autumn,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
57104,2025-11-30 21:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,84,941.2,3.9,5.0,92.0,84.0,92.0,943.9,...,0.5,autumn,-0.707107,7.071068e-01,-0.866025,0.5,-0.781831,0.62349,1,1
57105,2025-11-30 22:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,86,941.2,3.5,3.8,93.0,86.0,93.0,943.9,...,0.4,autumn,-0.500000,8.660254e-01,-0.866025,0.5,-0.781831,0.62349,1,1


In [28]:
future_df = TimeSeriesDataFrame.from_data_frame(
    filtered_df,
    id_column=ID_COL,
    timestamp_column=TIME_COL
)

In [29]:
predictions = predictor.predict(data=train_data,known_covariates=future_df)

Model not specified in predict, will default to the model with the best validation score: Chronos2


In [30]:
print(predictions.head())

                                                               mean  \
item_id                              timestamp                        
16836a55-7140-43e2-9a63-56fac5cba714 2025-11-09 16:00:00  38.731834   
                                     2025-11-09 17:00:00  40.098709   
                                     2025-11-09 18:00:00  37.771862   
                                     2025-11-09 19:00:00  35.190342   
                                     2025-11-09 20:00:00  32.144737   

                                                                0.1  \
item_id                              timestamp                        
16836a55-7140-43e2-9a63-56fac5cba714 2025-11-09 16:00:00  26.752377   
                                     2025-11-09 17:00:00  24.982611   
                                     2025-11-09 18:00:00  22.674049   
                                     2025-11-09 19:00:00  20.068165   
                                     2025-11-09 20:00:00  17.734859   

    

In [31]:
performance = predictor.evaluate(test_data)

print(performance)

Model not specified in predict, will default to the model with the best validation score: Chronos2


{'MASE': -0.9024280952131684}


In [32]:
test_df = test_data.to_data_frame()
pred_df = predictions.to_data_frame()

In [33]:
merged = test_df.merge(
    pred_df[["mean"]],
    on=["item_id", "timestamp"],
    how="inner"
)

In [34]:
merged.columns

Index(['humidity', 'pm10', 'pm25', 'pressure', 'temperature', 'wind_speed',
       'neighbor1_humidity', 'neighbor2_humidity', 'neighbor3_humidity',
       'neighbor1_pressure', 'neighbor2_pressure', 'neighbor3_pressure',
       'neighbor1_temperature', 'neighbor2_temperature',
       'neighbor3_temperature', 'neighbor1_wind_speed', 'neighbor2_wind_speed',
       'neighbor3_wind_speed', 'season', 'hour_sin', 'hour_cos', 'month_sin',
       'month_cos', 'day_sin', 'day_cos', 'is_weekend', 'is_heating_season',
       'mean'],
      dtype='object')

In [35]:
merged = merged.rename(columns={"mean": "predicted"})

In [36]:
merged

humidity       pm10  \
item_id                              timestamp                                  
16836a55-7140-43e2-9a63-56fac5cba714 2025-11-09 16:00:00     71.25  44.500000   
                                     2025-11-09 17:00:00     72.00  32.333333   
                                     2025-11-09 18:00:00     73.00  26.750000   
                                     2025-11-09 19:00:00     73.75  10.500000   
                                     2025-11-09 20:00:00     73.25   9.500000   
...                                                            ...        ...   
fec52a19-9148-4350-a1b4-ae0da05ee199 2025-11-30 19:00:00     61.75  11.250000   
                                     2025-11-30 20:00:00     63.25   9.000000   
                                     2025-11-30 21:00:00     63.50  12.000000   
                                     2025-11-30 22:00:00     63.75   3.250000   
                                     2025-11-30 23:00:00     63.75   1.750000   

                                                               pm25  pressure  \
item_id                              timestamp                                  
16836a55-7140-43e2-9a63-56fac5cba714 2025-11-09 16:00:00  29.750000    944.00   
                                     2025-11-09 17:00:00  20.666667    944.00   
                                     2025-11-09 18:00:00  16.750000    944.25   
                                     2025-11-09 19:00:00   6.250000    945.00   
                                     2025-11-09 20:00:00   6.250000    945.00   
...                                                             ...       ...   
fec52a19-9148-4350-a1b4-ae0da05ee199 2025-11-30 19:00:00   6.000000    942.00   
                                     2025-11-30 20:00:00   5.666667    942.00   
                                     2025-11-30 21:00:00   6.000000    942.50   
                                     2025-11-30 22:00:00   2.500000    943.00   
                                     2025-11-30 23:00:00   0.750000    943.00   

                                                          temperature  \
item_id                              timestamp                          
16836a55-7140-43e2-9a63-56fac5cba714 2025-11-09 16:00:00        16.00   
                                     2025-11-09 17:00:00        16.00   
                                     2025-11-09 18:00:00        15.75   
                                     2025-11-09 19:00:00        15.00   
                                     2025-11-09 20:00:00        15.00   
...                                                               ...   
fec52a19-9148-4350-a1b4-ae0da05ee199 2025-11-30 19:00:00         9.00   
                                     2025-11-30 20:00:00         9.00   
                                     2025-11-30 21:00:00         8.25   
                                     2025-11-30 22:00:00         7.75   
                                     2025-11-30 23:00:00         7.00   

                                                          wind_speed  \
item_id                              timestamp                         
16836a55-7140-43e2-9a63-56fac5cba714 2025-11-09 16:00:00         5.1   
                                     2025-11-09 17:00:00         5.6   
                                     2025-11-09 18:00:00         1.5   
                                     2025-11-09 19:00:00         4.5   
                                     2025-11-09 20:00:00         4.0   
...                                                              ...   
fec52a19-9148-4350-a1b4-ae0da05ee199 2025-11-30 19:00:00         5.9   
                                     2025-11-30 20:00:00         5.7   
                                     2025-11-30 21:00:00         5.8   
                                     2025-11-30 22:00:00         5.6   
                                     2025-11-30 23:00:00         6.2   

                                                          neigh

In [37]:
train_data

humidity  \
item_id                              timestamp                        
16836a55-7140-43e2-9a63-56fac5cba714 2023-12-01 00:00:00  58.333333   
                                     2023-12-01 01:00:00  59.250000   
                                     2023-12-01 02:00:00  60.000000   
                                     2023-12-01 03:00:00  59.250000   
                                     2023-12-01 04:00:00  59.500000   
...                                                             ...   
fec52a19-9148-4350-a1b4-ae0da05ee199 2025-11-09 11:00:00  57.000000   
                                     2025-11-09 12:00:00  59.750000   
                                     2025-11-09 13:00:00  64.666667   
                                     2025-11-09 14:00:00  66.000000   
                                     2025-11-09 15:00:00  67.500000   

                                                               pm10  \
item_id                              timestamp                        
16836a55-7140-43e2-9a63-56fac5cba714 2023-12-01 00:00:00  28.333333   
                                     2023-12-01 01:00:00  13.750000   
                                     2023-12-01 02:00:00  10.000000   
                                     2023-12-01 03:00:00   9.750000   
                                     2023-12-01 04:00:00   5.000000   
...                                                             ...   
fec52a19-9148-4350-a1b4-ae0da05ee199 2025-11-09 11:00:00   4.750000   
                                     2025-11-09 12:00:00   5.500000   
                                     2025-11-09 13:00:00   8.333333   
                                     2025-11-09 14:00:00  25.000000   
                                     2025-11-09 15:00:00  18.750000   

                                                               pm25  pressure  \
item_id                              timestamp                                  
16836a55-7140-43e2-9a63-56fac5cba714 2023-12-01 00:00:00  13.666667     943.0   
                                     2023-12-01 01:00:00   5.000000     943.0   
                                     2023-12-01 02:00:00   4.500000     942.5   
                                     2023-12-01 03:00:00   4.500000     942.0   
                                     2023-12-01 04:00:00   2.000000     942.0   
...                                                             ...       ...   
fec52a19-9148-4350-a1b4-ae0da05ee199 2025-11-09 11:00:00   1.000000     944.0   
                                     2025-11-09 12:00:00   2.750000     944.0   
                                     2025-11-09 13:00:00   3.333333     944.0   
                                     2025-11-09 14:00:00  10.250000     944.0   
                                     2025-11-09 15:00:00   8.250000     944.0   

                                                          temperature  \
item_id                              timestamp                          
16836a55-7140-43e2-9a63-56fac5cba714 2023-12-01 00:00:00        12.00   
                                     2023-12-01 01:00:00        12.00   
                                     2023-12-01 02:00:00        12.25   
                                     2023-12-01 03:00:00        13.00   
                                     2023-12-01 04:00:00        13.00   
...                                                               ...   
fec52a19-9148-4350-a1b4-ae0da05ee199 2025-11-09 11:00:00        18.50   
                                     2025-11-09 12:00:00        18.00   
                                     2025-11-09 13:00:00        17.00   
                                     2025-11-09 14:00:00        17.00   
                                     2025-11-09 15:00:00        16.50   

                                                          wind_speed  \
item_id                              timestamp                         
16836a55-7140-43e2-9a63-56fac5cba714 2023-12-01 00:00:00       

In [38]:

from pathlib import Path
import sqlite3

DB_PATH = Path("../data/bitola.db")
if not DB_PATH.exists():
    DB_PATH = Path("data/bitola.db")

if not DB_PATH.exists():
    raise FileNotFoundError("Could not find data/bitola.db. Run the notebook from offline-Phase or the project root.")

CITY = "Bitola"
MODEL_VERSION = f"chronos2_{TARGET}_bitola_offline_test_{PREDICTION_LENGTH}h"
MODEL_TYPE = 'fine_tuned'
train_df = (
    train_data.reset_index().rename(
        columns={
            "item_id": "sensor_id",
            TARGET: "actual_value"
        }
    )
)
train_df['predicted_value'] = np.nan
test_df = (
    merged.reset_index()
    .rename(columns={
        "item_id": "sensor_id",
        TARGET: "actual_value",
        "predicted": "predicted_value",
    })
)
offline_results = pd.concat([train_df,test_df],ignore_index=True).sort_values(["sensor_id", "timestamp"])
offline_results = offline_results[["sensor_id", "timestamp", "actual_value", "predicted_value"]].copy()
offline_results["city"] = CITY
offline_results["pollutant"] = TARGET
offline_results["model_version"] = MODEL_VERSION
offline_results["model_type"] = MODEL_TYPE
offline_results["timestamp"] = pd.to_datetime(offline_results["timestamp"]).dt.strftime("%Y-%m-%d %H:%M:%S")
offline_results = offline_results[["city", "sensor_id", "timestamp", "pollutant", "actual_value", "predicted_value", "model_version","model_type"]]

records = list(offline_results.itertuples(index=False, name=None))

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS offline_test_results (
            city TEXT NOT NULL,
            sensor_id TEXT NOT NULL,
            timestamp TEXT NOT NULL,
            pollutant TEXT NOT NULL,
            actual_value REAL,
            predicted_value REAL,
            model_version TEXT NOT NULL,
            model_type TEXT NOT NULL,
            PRIMARY KEY (city, sensor_id, timestamp, pollutant, model_version)
        )
    """)
    conn.executemany("""
        INSERT OR REPLACE INTO offline_test_results (
            city, sensor_id, timestamp, pollutant, actual_value, predicted_value, model_version,model_type
        ) VALUES (?, ?, ?, ?, ?, ?, ?,?)
    """, records)

print(f"Saved {len(records)} {CITY} {TARGET} offline test rows to {DB_PATH}")

/tmp/ipykernel_347962/3114147318.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  offline_results = pd.concat([train_df,test_df],ignore_index=True).sort_values(["sensor_id", "timestamp"])


Saved 192984 Bitola pm10 offline test rows to ../data/bitola.db


In [39]:
merged.dropna(inplace=True)

In [40]:
y_true = merged["pm10"]
y_pred = merged["predicted"]

r2_global = r2_score(y_true, y_pred)
rmse_global = np.sqrt(mean_squared_error(y_true, y_pred))

print("Global R2:", r2_global)
print("Global RMSE:", rmse_global)

Global R2: 0.2548877297292158
Global RMSE: 27.785316206336375


In [41]:
per_sensor = merged.groupby("item_id").apply(
    lambda df: pd.Series({
        "r2": r2_score(df["pm10"], df["predicted"]),
        "rmse": np.sqrt(mean_squared_error(df["pm10"], df["predicted"]))
    })
)

print(per_sensor)

                                            r2       rmse
item_id                                                  
16836a55-7140-43e2-9a63-56fac5cba714  0.120774  12.987241
2002                                  0.145305  23.901433
23b735ef-a996-4a7f-9998-2aa7e78827b0  0.090245  28.649348
2819ecbb-5de3-4092-aa1d-4ba3a8c40add  0.086338  10.687351
40f081a6-4095-43f7-bffb-64e2af8c026e -0.024724  22.187034
7b316592-8036-41e2-b8dc-b06b6a9afd54  0.249347  43.955149
874ff9c6-786d-45fc-a90e-48c7ffe03417  0.044830  31.797033
87f82783-853b-417d-8964-b5cf11e44873  0.169928  47.388756
d241a044-0a06-40c2-9d90-c91fd0a95060 -0.022858  11.010461
d7060523-b163-4cc3-bc94-970dac72a38a  0.062755  29.843329
fec52a19-9148-4350-a1b4-ae0da05ee199  0.176700   7.989832


In [42]:
predictor.path

'/mnt/c/Users/RazorVision/Desktop/project-vrnmp/offline-Phase/AutogluonModels/ag-20260715_114407'

In [43]:
import shutil

shutil.copytree(
    predictor.path,
    "chronos2_model_pm10_bitola"
)

'chronos2_model_pm10_bitola'